<a href="https://colab.research.google.com/github/jordanguali/AplicacionWeb1NominaISLinea/blob/hector/SparkSession_RDDs_TAREA_BD_AFPG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Instalar Java
# ! -> entro en el cmd de linux y explicar que hace

! apt-get install openjdk-8-jdk-headless -qq > /dev/null

#Bajar, descompactar y configurar el Apache Spark
#Ver donde guardo -> carpeta

! wget -q https://dlcdn.apache.org/spark/spark-3.5.4/spark-3.5.4-bin-hadoop3.tgz

#Descompactar
! tar -xf spark-3.5.4-bin-hadoop3.tgz

#Eliminar el .tgz
! rm -rf spark-3.5.4-bin-hadoop3.tgz

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
#donde esta colocado - ver con click derecho donde esta el ambiente
os.environ["SPARK_HOME"] = "/content/spark-3.5.4-bin-hadoop3"

## Apache Spark en versiones inferiores a la 2.0

In [ ]:
from pyspark import SparkConf, SparkContext

In [ ]:
conf = SparkConf().setAppName("MiApp").setMaster("local[*]")

In [ ]:
sc = SparkContext(conf = conf)

## Apache Spark en versiones superiores a la 2.0

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("Exercicio01").getOrCreate()

In [ ]:
# Creacion de Dataframe en el Spark
data = [("Luis", 34), ("Maria", 25), ("Roberto", 31)]
columnas = ["Nombre", "Edad"]

In [ ]:
df = spark.createDataFrame(data, columnas)

In [ ]:
df.show()

+-------+----+
| Nombre|Edad|
+-------+----+
|   Luis|  34|
|  Maria|  25|
|Roberto|  31|
+-------+----+



In [ ]:
# Ejemplo de uso de consulta SQL
df.createOrReplaceTempView("personas")
spark.sql("SELECT * FROM personas WHERE Edad > 30").show()

+-------+----+
| Nombre|Edad|
+-------+----+
|   Luis|  34|
|Roberto|  31|
+-------+----+



## RDDs

In [1]:
from pyspark.sql import SparkSession

In [3]:
# Crear session de Spark
spark = SparkSession.builder.appName("RDD_Ejemplo").getOrCreate()

In [4]:
# Crear un RDD desde una lista
numeros = [1, 2, 3, 4, 5]
rdd = spark.sparkContext.parallelize(numeros)

In [5]:
rdd

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:289

In [6]:
# Mostrar el contenido del RDD
print("Datos originales: ", rdd.collect())

## NOTA IMPORTANTE: collect() debe usarse con cuidado en grandes volumenes de datos
## ya que trae todos los datos a la memora.
## En produccion es mejor usar acciones como take(n) para ver solo algunas muestras.

Datos originales:  [1, 2, 3, 4, 5]


In [7]:
# Especificando el numero de particiones
rdd2 = spark.sparkContext.parallelize(numeros, 3)

In [8]:
print("Numero de particiones del rdd2: ", rdd2.getNumPartitions())

Numero de particiones del rdd2:  3


In [9]:
# El contenido de cada particion
print("Contenido de cada particion: ", rdd2.glom().collect())

Contenido de cada particion:  [[1], [2, 3], [4, 5]]


### Creacion de un RDD y Operaciones Basicas

In [10]:
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
rdd = spark.sparkContext.parallelize(data)

In [11]:
# Transformaciones basicas
rdd_cuadrados = rdd.map(lambda x: x ** 2)
rdd_cuadrados.collect()

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]

In [ ]:
def suma(a, b):
  return a + b

In [ ]:
resultado = suma(5, 3)
resultado

8

In [ ]:
suma1 = lambda a, b: a + b

In [ ]:
resultadoLamba = suma1(3, 5)
resultadoLamba

8

In [ ]:
rdd_filtrado = rdd_cuadrados.filter(lambda x: x > 10)
rdd_filtrado.collect()

[16, 25, 36, 49, 64, 81, 100]

## Conteo de Palabras

In [12]:
frase = ["Hola Mundo de PySpark", "Bienvenidos al PySpark", "Creando RDDs", "Contando RDDs"]
rdd_texto = spark.sparkContext.parallelize(frase, 3)

In [13]:
# Transformaciones
rdd_palabras = rdd_texto.flatMap(lambda x: x.split(" ")) #Dividiendo cada frase en palabras
rdd_asignar = rdd_palabras.map(lambda x: (x, 1)) #Asignando un valor de 1 a cada palabra
rdd_contando = rdd_asignar.reduceByKey(lambda a,b : a+b) #Sumando las apariciones de cada palabra

In [14]:
rdd_contando.collect()

[('de', 1),
 ('PySpark', 2),
 ('al', 1),
 ('Creando', 1),
 ('Hola', 1),
 ('Mundo', 1),
 ('Bienvenidos', 1),
 ('RDDs', 2),
 ('Contando', 1)]

## Acciones sobre RDDs

In [15]:
#Reduce -> agregar (reduciend) los elementos de un RDD en un solo valor.
#rdd.reduce(func) -> realizar la reduccion a un solo valor.

sc = spark.sparkContext

rdd = sc.parallelize([2, 5, 7, 1, 2, 6])

In [16]:
suma = rdd.reduce(lambda x, y: x + y)
print(suma)

# (2 + 5) = 7
# (7 + 7) = 14
# (14 + 1) = 15
# (15 + 2) = 17
# (17 + 6) = 23

23


In [17]:
maximo_valor = rdd.reduce(lambda x, y: x if x > y else y)
print(maximo_valor)

# (2, 5) = 5
# (5, 7) = 7
# (7, 1) = 7
# (7, 2) = 7
# (7, 6) = 7

7


In [18]:
#Count -> conteo de los elementos del RDD
rdd.collect()

[2, 5, 7, 1, 2, 6]

In [19]:
print(rdd.count())

6


In [20]:
#Collect -> Devuelve los elementos del RDD a la memoria del driver (driver del Apache Spark)

In [21]:
#First -> obtiene el primer elemento del conjunto de datos distribuidos
#rdd.first() -> No recibe argumentos.
primero = rdd.first()
print(primero)

2


In [22]:
#Take -> devolver los primeros n elementos del RDD
#rdd.take(n) -> n es los primeros n elementos
print(rdd.take(3))

[2, 5, 7]


In [23]:
#ForEach -> en un RDD es usado para aplicar una funcion a cada elemento sin devolver ningun valor al driver program.
#rdd.foreach(func) -> func es la funcion que se aplica a cada elemento del RDD
#No devuelve nada.

rdd.collect()

[2, 5, 7, 1, 2, 6]

In [24]:
rdd.foreach(lambda x: print(x)) # No vamos a ver nada a la salida en clusters distribuidos

In [25]:
#saveAsTextFile -> usamos en RDDs para guardar los datos en formato de texto.
#rdd.saveAsTextFile("ruta/del/directorio") -> es el directorio donde se almacenaran los archivos de texto
rdd.saveAsTextFile("lista")

## Actividad

1. Crear un RDD llamado importes a partir del archivo adjunto a esta lección como recurso.

2. ¿Cuántos registros tiene el RDD importes?

3. ¿Cuál es el valor mínimo y máximo del RDD importes?

4. Cree un RDD top15 que contenga los 15 mayores valores del RDD importes. Tenga en cuenta que pueden repetirse los valores. Por último, escriba el RDD top15 como archivo de texto en la carpeta content.

In [32]:
#1.

importes = sc.textFile('/content/num.txt')

In [36]:
# Crear session de Spark
spark = SparkSession.builder.appName("RDD_Importes").getOrCreate()


In [46]:
# Crear un RDD desde una lista
rdd_importes = sc.textFile("/content/num.txt")

In [47]:
cantidad_registros = rdd_importes.count()
print(f"El RDD importes tiene {cantidad_registros} registros.")

El RDD importes tiene 233 registros.


In [48]:
valor_minimo = rdd_importes.min()
valor_maximo = rdd_importes.max()

print(f"El valor mínimo en el RDD importes es: {valor_minimo}")
print(f"El valor máximo en el RDD importes es: {valor_maximo}")


El valor mínimo en el RDD importes es: 1
El valor máximo en el RDD importes es: 99


In [61]:
# prompt: Cree un RDD top15 que contenga los 15 mayores valores del RDD importes. Tenga en cuenta que pueden repetirse los valores. Por último, escriba el RDD top15 como archivo de texto en la carpeta content.

# Convertir el RDD de texto a números
rdd_numeros = importes.map(lambda x: int(x))

# Obtener los 15 mayores valores
top15 = rdd_numeros.top(15)

# Crear un nuevo RDD con los 15 mayores valores
rdd_top15 = sc.parallelize(top15)

# Guardar el RDD top15 como archivo de texto
rdd_top15.saveAsTextFile("content/top15_1")
